Step 1 Install libraries

In [1]:
!pip -q install -U google-genai gradio

Step 2 import libraries

In [2]:
import pandas as pd
import requests
import json
import os

from PIL import Image
from getpass import getpass

import gradio as gr

from google import genai

 Step 3 Configure API

In [37]:
GEMINI_API_KEY = getpass("Enter your Gemini API Key: ")

client = genai.Client(api_key=GEMINI_API_KEY)

MODEL_NAME = "gemini-3.5-flash-lite"

print("Gemini client configured successfully.")

Enter your Gemini API Key: ··········
Gemini client configured successfully.


In [38]:
try:
    test_response = client.models.generate_content(
        model=MODEL_NAME,
        contents="Create a one sentence travel tip for Goa."
    )

    print("SUCCESS!")
    print(test_response.text)

except Exception as e:
    print(type(e).__name__)
    print(str(e))

SUCCESS!
Rent a scooter to easily navigate between the bustling northern beaches and the tranquil southern shores, but always wear a helmet and drive cautiously.


 step 4 Destination dataset



In [5]:
data = {
    "destination": [
        "Bengaluru",
        "Goa",
        "Mysuru",
        "Ooty",
        "Manali",
        "Jaipur",
        "Munnar",
        "Coorg"
    ],

    "state": [
        "Karnataka",
        "Goa",
        "Karnataka",
        "Tamil Nadu",
        "Himachal Pradesh",
        "Rajasthan",
        "Kerala",
        "Karnataka"
    ],

    "type": [
        "City",
        "Beach",
        "Heritage",
        "Hill Station",
        "Mountain",
        "Heritage",
        "Hill Station",
        "Nature"
    ],

    "interests": [
        "Food, Technology, Parks, History",
        "Beach, Food, Nightlife, Water Sports",
        "History, Palace, Culture, Food",
        "Nature, Mountains, Tea, Relaxation",
        "Mountains, Adventure, Snow, Nature",
        "History, Culture, Forts, Shopping",
        "Tea, Nature, Mountains, Waterfalls",
        "Coffee, Nature, Trekking, Relaxation"
    ]
}

destination_df = pd.DataFrame(data)

display(destination_df)

,destination,state,type,interests
0,Bengaluru,Karnataka,City,"Food, Technology, Parks, History"
1,Goa,Goa,Beach,"Beach, Food, Nightlife, Water Sports"
2,Mysuru,Karnataka,Heritage,"History, Palace, Culture, Food"
3,Ooty,Tamil Nadu,Hill Station,"Nature, Mountains, Tea, Relaxation"
4,Manali,Himachal Pradesh,Mountain,"Mountains, Adventure, Snow, Nature"
5,Jaipur,Rajasthan,Heritage,"History, Culture, Forts, Shopping"
6,Munnar,Kerala,Hill Station,"Tea, Nature, Mountains, Waterfalls"
7,Coorg,Karnataka,Nature,"Coffee, Nature, Trekking, Relaxation"


Step 5 Data cleaning

In [6]:
# Remove duplicate rows
destination_df = destination_df.drop_duplicates()

# Remove rows with missing destination names
destination_df = destination_df.dropna(
    subset=["destination"]
)

# Remove unnecessary spaces
destination_df["destination"] = (
    destination_df["destination"].str.strip()
)

destination_df["state"] = (
    destination_df["state"].str.strip()
)

destination_df["type"] = (
    destination_df["type"].str.strip()
)

print("Dataset cleaned successfully.")

display(destination_df)

Dataset cleaned successfully.


,destination,state,type,interests
0,Bengaluru,Karnataka,City,"Food, Technology, Parks, History"
1,Goa,Goa,Beach,"Beach, Food, Nightlife, Water Sports"
2,Mysuru,Karnataka,Heritage,"History, Palace, Culture, Food"
3,Ooty,Tamil Nadu,Hill Station,"Nature, Mountains, Tea, Relaxation"
4,Manali,Himachal Pradesh,Mountain,"Mountains, Adventure, Snow, Nature"
5,Jaipur,Rajasthan,Heritage,"History, Culture, Forts, Shopping"
6,Munnar,Kerala,Hill Station,"Tea, Nature, Mountains, Waterfalls"
7,Coorg,Karnataka,Nature,"Coffee, Nature, Trekking, Relaxation"


Step 6 destination filtering

In [7]:
def get_destination_info(destination):

    result = destination_df[
        destination_df["destination"].str.lower()
        == destination.lower().strip()
    ]

    if len(result) == 0:
        return None

    return result.iloc[0].to_dict()

Step 7 Weather API

In [8]:
def get_weather(destination):

    # -----------------------------
    # STEP 1: Get coordinates
    # -----------------------------

    geo_url = "https://geocoding-api.open-meteo.com/v1/search"

    geo_params = {
        "name": destination,
        "count": 1,
        "language": "en",
        "format": "json"
    }

    geo_response = requests.get(
        geo_url,
        params=geo_params,
        timeout=10
    )

    geo_response.raise_for_status()

    geo_data = geo_response.json()

    if "results" not in geo_data:
        return None

    location = geo_data["results"][0]

    latitude = location["latitude"]
    longitude = location["longitude"]

    # -----------------------------
    # STEP 2: Get weather
    # -----------------------------

    weather_url = "https://api.open-meteo.com/v1/forecast"

    weather_params = {
        "latitude": latitude,
        "longitude": longitude,
        "current": "temperature_2m,relative_humidity_2m,weather_code",
        "daily": "temperature_2m_max,temperature_2m_min,precipitation_probability_max",
        "forecast_days": 3,
        "timezone": "auto"
    }

    weather_response = requests.get(
        weather_url,
        params=weather_params,
        timeout=10
    )

    weather_response.raise_for_status()

    weather_data = weather_response.json()

    return weather_data

Step 8 Weather Data

In [9]:
def simplify_weather(weather):

    if weather is None:
        return {
            "status": "Weather information unavailable."
        }

    current = weather.get("current", {})
    daily = weather.get("daily", {})

    result = {
        "current_temperature": current.get(
            "temperature_2m"
        ),

        "humidity": current.get(
            "relative_humidity_2m"
        ),

        "dates": daily.get(
            "time", []
        ),

        "maximum_temperature": daily.get(
            "temperature_2m_max", []
        ),

        "minimum_temperature": daily.get(
            "temperature_2m_min", []
        ),

        "rain_probability": daily.get(
            "precipitation_probability_max", []
        )
    }

    return result

Step 9 Multimodel image

In [10]:
def analyze_destination_image(image):

    if image is None:
        return "No destination image was provided."

    prompt = """
You are a travel image analysis assistant.

Analyze the uploaded destination image.

Identify:

1. The type of place visible.
2. Any recognizable landmark or tourist attraction.
3. Possible tourist activities.
4. What kind of traveler may enjoy this place.
5. Whether the image gives any useful travel information.

Important:
- Do not invent an exact location if the image does not provide enough evidence.
- If the exact location cannot be identified, clearly say that the location is uncertain.
- Keep the answer concise.
"""

    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=[
            image,
            prompt
        ]
    )

    return response.text

Step 10 Conversation memory

In [11]:
conversation_memory = []

In [12]:
def add_to_memory(
    destination,
    days,
    budget,
    travel_style,
    interests
):

    memory_item = {
        "destination": destination,
        "days": days,
        "budget": budget,
        "travel_style": travel_style,
        "interests": interests
    }

    conversation_memory.append(memory_item)

    # Keep only the latest 5 requests
    if len(conversation_memory) > 5:
        conversation_memory.pop(0)

Step 11 Context Engineering

In [13]:
def build_context(
    destination,
    days,
    budget,
    travel_style,
    interests,
    extra_request,
    image_analysis
):

    # -----------------------------
    # STATIC CONTEXT
    # -----------------------------

    destination_info = get_destination_info(
        destination
    )

    # -----------------------------
    # DYNAMIC CONTEXT
    # -----------------------------

    weather_data = get_weather(
        destination
    )

    simple_weather = simplify_weather(
        weather_data
    )

    # -----------------------------
    # MEMORY CONTEXT
    # -----------------------------

    previous_requests = conversation_memory.copy()

    # Add current request to memory
    add_to_memory(
        destination,
        days,
        budget,
        travel_style,
        interests
    )

    # -----------------------------
    # FINAL CONTEXT
    # -----------------------------

    context = {

        "STATIC_CONTEXT": {
            "destination_database": destination_info
        },

        "DYNAMIC_CONTEXT": {
            "weather": simple_weather
        },

        "USER_CONTEXT": {
            "trip_duration": days,
            "budget": budget,
            "travel_style": travel_style,
            "interests": interests,
            "additional_request": extra_request
        },

        "IMAGE_CONTEXT": {
            "image_analysis": image_analysis
        },

        "MEMORY_CONTEXT": {
            "previous_requests": previous_requests
        }
    }

    return context

Step 12 Prompt Engineering

In [14]:
def create_prompt(context):

    prompt = f"""
ROLE:
You are an intelligent AI Travel Planner.

TASK:
Create a personalized travel itinerary using
destination information, weather, user preferences,
destination image analysis and previous conversation context.

==============================
STATIC CONTEXT
==============================

{json.dumps(
    context["STATIC_CONTEXT"],
    indent=2
)}

==============================
DYNAMIC CONTEXT
==============================

{json.dumps(
    context["DYNAMIC_CONTEXT"],
    indent=2
)}

==============================
USER CONTEXT
==============================

{json.dumps(
    context["USER_CONTEXT"],
    indent=2
)}

==============================
IMAGE CONTEXT
==============================

{json.dumps(
    context["IMAGE_CONTEXT"],
    indent=2
)}

==============================
MEMORY CONTEXT
==============================

{json.dumps(
    context["MEMORY_CONTEXT"],
    indent=2
)}

==============================
CONSTRAINTS
==============================

1. Respect the user's budget.
2. Respect the number of travel days.
3. Consider the weather information.
4. Match activities with the user's interests.
5. Avoid unrealistic schedules.
6. Do not invent weather information.
7. Do not claim that an image location is certain unless
   the image provides enough evidence.
8. Keep the recommendations practical.
9. Use simple language.

==============================
OUTPUT FORMAT
==============================

## Trip Summary

Destination:
Duration:
Travel Style:
Budget:

## Day 1

Morning:
Afternoon:
Evening:

## Day 2

Morning:
Afternoon:
Evening:

## Day 3

Morning:
Afternoon:
Evening:

## Estimated Budget

Transport:
Food:
Activities:
Accommodation:
Other:

## Weather Advice

## Travel Tips

Give a clear and useful travel plan.
"""

    return prompt

Step 13 Travel plan

In [34]:
def generate_travel_plan(
    destination,
    days,
    budget,
    travel_style,
    interests,
    extra_request,
    image
):
    destination_info = get_destination_info(destination)

    if destination_info is None:
        return (
            f"Sorry, **{destination}** is not currently "
            "available in our destination dataset.\n\n"
            "Try one of these:\n\n"
            + "\n".join(
                f"- {x}"
                for x in destination_df["destination"]
            )
        )

    image_analysis = analyze_destination_image(image)

    context = build_context(
        destination,
        days,
        budget,
        travel_style,
        interests,
        extra_request,
        image_analysis
    )

    prompt = create_prompt(context)

    try:
        response = client.models.generate_content(
            model=MODEL_NAME,
            contents=prompt
        )

        return response.text

    except Exception as e:

        if "503" in str(e) or "UNAVAILABLE" in str(e):

            print("Gemini 3.6 Flash is busy.")
            print("Trying fallback model...")

            response = client.models.generate_content(
                model="gemini-3.5-flash-lite",
                contents=prompt
            )

            return response.text

        else:
            raise

Step 14 Testing

In [39]:
try:
    result = generate_travel_plan(
        destination="Goa",
        days=3,
        budget="₹10000",
        travel_style="Budget",
        interests="Beach, Food, Nature",
        extra_request="I prefer less crowded places",
        image=None
    )

    print(result)

except Exception as e:
    print("===================================")
    print("ERROR DETAILS")
    print("===================================")
    print(type(e).__name__)
    print(str(e))

## Trip Summary

Destination: Goa
Duration: 3 Days
Travel Style: Budget
Budget: ₹10,000

## Day 1

Morning: Arrive and check into a budget hostel or guesthouse in North Goa. Head straight to the relatively quiet **Morjim Beach** to enjoy a peaceful morning walk and watch the waves without heavy crowds.
Afternoon: Enjoy a budget-friendly Goan fish thali or local vegetarian meal at a beachside local shack.
Evening: Visit the serene **Chapora Fort** for a scenic, breezy sunset view of the coastline and surrounding nature.

## Day 2

Morning: Explore the lush nature trails around **Cotigao Wildlife Sanctuary** or take a peaceful morning stroll at **Cola Beach**, known for its hidden freshwater lagoon meeting the sea.
Afternoon: Have lunch at a local cafe or dhaba, trying out traditional Goan snacks and fresh lime soda.
Evening: Relax on the peaceful sands of **Arambol Beach**, watch the sunset, and explore the laid-back local market area.

## Day 3

Morning: *Weather Note: Rain is expected

 Step 15 Gradio Application

In [40]:
with gr.Blocks(
    title="AI Smart Travel Planner"
) as demo:

    gr.Markdown(
        """
        # ✈️ AI Smart Travel Planner

        ### Personalized travel planning using
        destination data, weather, user preferences
        and destination image understanding.
        """
    )

    with gr.Row():

        # ==========================
        # LEFT SIDE - USER INPUT
        # ==========================

        with gr.Column():

            destination = gr.Textbox(
                label="Destination",
                placeholder="Example: Goa"
            )

            days = gr.Number(
                label="Number of Days",
                value=3,
                minimum=1,
                maximum=3
            )

            budget = gr.Textbox(
                label="Budget",
                placeholder="Example: ₹10000"
            )

            travel_style = gr.Dropdown(
                choices=[
                    "Budget",
                    "Luxury",
                    "Family",
                    "Adventure",
                    "Relaxed"
                ],
                label="Travel Style",
                value="Budget"
            )

            interests = gr.Textbox(
                label="Interests",
                placeholder="Example: Beach, Food, Nature"
            )

            extra_request = gr.Textbox(
                label="Additional Request",
                placeholder="Example: Avoid crowded places"
            )

            image = gr.Image(
                type="pil",
                label="Upload Destination Image"
            )

            generate_button = gr.Button(
                "✈️ Generate Travel Plan"
            )

        # ==========================
        # RIGHT SIDE - OUTPUT
        # ==========================

        with gr.Column():

            output = gr.Markdown(
                value="Your travel plan will appear here."
            )

    # ==========================
    # BUTTON ACTION
    # ==========================

    generate_button.click(
        fn=generate_travel_plan,
        inputs=[
            destination,
            days,
            budget,
            travel_style,
            interests,
            extra_request,
            image
        ],
        outputs=output
    )


demo.launch(
    share=True,
    debug=True
)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://6d77d6ac9733922ac6.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://6d77d6ac9733922ac6.gradio.live
